# Vertex AI Word2Vec + XGBoost — Simple End-to-End Test

This notebook follows the same overall pattern as the HSBC Confluence document:

**sample data → GCS → custom training package → Vertex AI CustomPythonPackageTrainingJob → model artifacts → Model Registry**

The sample dataset is created **inside this notebook**, so there is no separate CSV file to upload manually.

The dataset has exactly three columns:

- `ID` — identifier only; not used as a feature
- `text` — text used to train Word2Vec
- `target` — classification label used by XGBoost

This is a **pipeline smoke test**, not a meaningful model-performance evaluation.


## 1. Configuration

Change only these values.

The CMEK values below are copied from the Confluence documentation you provided.


In [ ]:
# ============================================================
# CHANGE ONLY THESE VALUES
# ============================================================

PROJECT_ID = "YOUR_GCP_PROJECT_ID"

REGION = "europe-west2"

BUCKET = "gs://YOUR_USE_CASE_BUCKET"

USE_CASE_SERVICE_ACCOUNT = "YOUR_USE_CASE_SERVICE_ACCOUNT"

# Same CMEK shown in the Confluence documentation
CMEK = (
    "projects/hsbc-6320774-kms-dev/"
    "locations/europe-west2/"
    "keyRings/VertexAI/"
    "cryptoKeys/vtSharedKey"
)

# Same training/prediction container pattern shown in the document
TRAIN_IMAGE = (
    "europe-docker.pkg.dev/vertex-ai/training/"
    "xgboost-cpu.2-1:latest"
)

DEPLOY_IMAGE = (
    "europe-docker.pkg.dev/vertex-ai/prediction/"
    "xgboost-cpu.2-1:latest"
)

print("Configuration loaded.")


## 2. Install the notebook-side packages

The Vertex training job installs its own dependencies from `setup.py`.


In [ ]:
%pip install -q google-cloud-aiplatform pandas gensim xgboost scikit-learn


## 3. Imports and Vertex AI initialization

In [ ]:
import os
import json
import tarfile
from pathlib import Path

import pandas as pd
from google.cloud import aiplatform, storage

aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=BUCKET,
)

print("Vertex AI initialized.")


## 4. Create the 50-row sample dataset

This is deliberately small. We only want to prove that the complete pipeline works.


In [ ]:
sample_rows = [
    (1, "customer made a suspicious payment to an unknown beneficiary", 1),
    (2, "transaction was flagged for unusual activity and requires review", 1),
    (3, "customer requested an urgent transfer to a new account", 1),
    (4, "payment pattern indicates potentially suspicious activity", 1),
    (5, "account showed an unusual transfer inconsistent with normal behaviour", 1),
    (6, "customer attempted multiple high value transactions in a short period", 1),
    (7, "unusual payment activity was identified by the monitoring team", 1),
    (8, "beneficiary details appear inconsistent and transaction needs review", 1),
    (9, "customer activity triggered a potential regulatory concern", 1),
    (10, "large transfer was made to a newly added beneficiary", 1),
    (11, "transaction was escalated because of suspicious payment behaviour", 1),
    (12, "account activity contains an unusual sequence of payments", 1),
    (13, "customer requested a high value transfer with limited explanation", 1),
    (14, "payment was flagged due to unusual transaction frequency", 1),
    (15, "monitoring identified potentially risky customer behaviour", 1),
    (16, "transaction requires compliance review because of unusual activity", 1),
    (17, "customer sent funds to a beneficiary with unexpected details", 1),
    (18, "multiple unusual payments were detected on the account", 1),
    (19, "payment was escalated due to a potential policy breach", 1),
    (20, "customer activity appears inconsistent with the expected pattern", 1),
    (21, "large payment triggered an alert for suspicious activity", 1),
    (22, "unusual beneficiary activity requires further investigation", 1),
    (23, "transaction was identified as potentially high risk", 1),
    (24, "customer attempted an unusual transfer outside the normal pattern", 1),
    (25, "compliance monitoring flagged the transaction for review", 1),

    (26, "customer completed a normal payment to a regular beneficiary", 0),
    (27, "salary payment was received successfully into the account", 0),
    (28, "customer transferred funds between their own accounts", 0),
    (29, "routine monthly payment was processed successfully", 0),
    (30, "customer made a regular payment to an existing beneficiary", 0),
    (31, "account activity was consistent with the normal transaction pattern", 0),
    (32, "standard payment was completed without any issues", 0),
    (33, "customer received a routine transfer from an existing account", 0),
    (34, "regular bill payment was processed successfully", 0),
    (35, "customer completed a normal account transfer", 0),
    (36, "payment was successfully processed for a known beneficiary", 0),
    (37, "routine customer activity was observed during the review period", 0),
    (38, "customer made a standard transfer within the normal account pattern", 0),
    (39, "monthly account payment was completed successfully", 0),
    (40, "customer received a normal salary transaction", 0),
    (41, "regular transfer was completed between linked customer accounts", 0),
    (42, "standard payment activity was observed with no unusual behaviour", 0),
    (43, "customer made a routine payment to a known merchant", 0),
    (44, "normal account activity continued without any alerts", 0),
    (45, "customer completed a regular transfer using the existing beneficiary", 0),
    (46, "routine payment was processed successfully", 0),
    (47, "known beneficiary received a normal customer payment", 0),
    (48, "account activity remained consistent with previous transactions", 0),
    (49, "customer completed the expected monthly payment", 0),
    (50, "normal transaction activity was recorded on the account", 0),
]

df = pd.DataFrame(sample_rows, columns=["ID", "text", "target"])

DATA_FILE = "dataset.csv"
df.to_csv(DATA_FILE, index=False)

print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
print("\nTarget distribution:")
print(df["target"].value_counts())

display(df.head())


## 5. Upload the sample dataset to GCS

This corresponds to the data-upload part of the Confluence workflow.


In [ ]:
DATASET_URI = f"{BUCKET}/word2vec-test/data/dataset.csv"

!gsutil cp dataset.csv "$DATASET_URI"

print("Dataset uploaded to:", DATASET_URI)


## 6. Create the custom training package

This creates the same package structure used in the Confluence example:

```text
custom/
├── setup.py
└── trainer/
    ├── __init__.py
    └── task.py
```

`task.py` performs:

**CSV → tokenize text → Word2Vec → document vectors → XGBoost → metrics → model artifacts**


In [ ]:
# Create package folders
!rm -rf custom custom.tar.gz
!mkdir -p custom/trainer

# Package files


In [ ]:
%%writefile custom/setup.py
from setuptools import setup, find_packages

setup(
    name="word2vec-xgboost-training",
    version="0.1.0",
    packages=find_packages(),
    install_requires=[
        "gensim==4.3.3",
        "numpy>=1.26,<2.1",
        "pandas>=2.2,<3.0",
        "scikit-learn>=1.5,<2.0",
        "xgboost>=2.1,<3.0",
        "google-cloud-storage>=2.19,<3.0",
        "cloudml-hypertune>=0.1.0",
    ],
)


In [ ]:
%%writefile custom/trainer/__init__.py


In [ ]:
%%writefile custom/trainer/task.py
import argparse
import json
import logging
import os
import re
import tempfile

import numpy as np
import pandas as pd
import xgboost as xgb

from gensim.models import Word2Vec
from google.cloud import storage
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

try:
    import hypertune
except ImportError:
    hypertune = None


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--model-dir", required=True)
    parser.add_argument("--dataset-data-url", required=True)

    parser.add_argument("--vector-size", type=int, default=50)
    parser.add_argument("--window", type=int, default=3)
    parser.add_argument("--min-count", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=10)

    parser.add_argument("--xgb-rounds", type=int, default=30)

    return parser.parse_args()


def download_gcs(uri, local_file):
    path = uri.replace("gs://", "", 1)
    bucket_name, blob_name = path.split("/", 1)

    client = storage.Client()
    bucket = client.bucket(bucket_name)
    bucket.blob(blob_name).download_to_filename(local_file)


def tokenize(text):
    text = str(text).lower()
    return re.findall(r"\b\w+\b", text)


def document_vector(tokens, model):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if not vectors:
        return np.zeros(model.vector_size, dtype=np.float32)

    return np.mean(vectors, axis=0).astype(np.float32)


def main():
    args = parse_args()

    with tempfile.TemporaryDirectory() as tmp:

        dataset_file = os.path.join(tmp, "dataset.csv")

        logger.info("Downloading dataset.")
        download_gcs(
            args.dataset_data_url,
            dataset_file,
        )

        df = pd.read_csv(dataset_file)

        required = {"ID", "text", "target"}
        missing = required - set(df.columns)

        if missing:
            raise ValueError(f"Missing columns: {missing}")

        df["text"] = df["text"].fillna("").astype(str)
        df["target"] = df["target"].astype(int)

        # ID is metadata only.
        # text is the Word2Vec input.
        # target is the XGBoost label.

        train_idx, test_idx = train_test_split(
            np.arange(len(df)),
            test_size=0.20,
            random_state=42,
            stratify=df["target"],
        )

        train_sentences = [
            tokenize(df.iloc[i]["text"])
            for i in train_idx
        ]

        logger.info("Training Word2Vec.")
        w2v = Word2Vec(
            sentences=train_sentences,
            vector_size=args.vector_size,
            window=args.window,
            min_count=args.min_count,
            workers=1,
            sg=1,
            epochs=args.epochs,
            seed=42,
        )

        all_sentences = [
            tokenize(text)
            for text in df["text"]
        ]

        X = np.vstack([
            document_vector(tokens, w2v)
            for tokens in all_sentences
        ])

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = df.iloc[train_idx]["target"].values
        y_test = df.iloc[test_idx]["target"].values

        logger.info("Training XGBoost.")

        train_matrix = xgb.DMatrix(
            X_train,
            label=y_train,
        )

        test_matrix = xgb.DMatrix(X_test)

        model = xgb.train(
            {
                "objective": "binary:logistic",
                "eval_metric": "logloss",
                "max_depth": 4,
                "learning_rate": 0.1,
                "seed": 42,
            },
            train_matrix,
            num_boost_round=args.xgb_rounds,
        )

        probabilities = model.predict(test_matrix)
        predictions = (probabilities >= 0.5).astype(int)

        accuracy = accuracy_score(
            y_test,
            predictions,
        )

        f1 = f1_score(
            y_test,
            predictions,
            zero_division=0,
        )

        metrics = {
            "accuracy": float(accuracy),
            "f1": float(f1),
            "confusion_matrix": confusion_matrix(
                y_test,
                predictions,
            ).tolist(),
            "word2vec_vocabulary_size": len(w2v.wv),
            "word2vec_vector_size": args.vector_size,
        }

        output_dir = os.path.join(tmp, "model")
        os.makedirs(output_dir, exist_ok=True)

        # XGBoost model
        model.save_model(
            os.path.join(output_dir, "model.bst")
        )

        # Word2Vec model
        w2v.save(
            os.path.join(output_dir, "word2vec.model")
        )

        # Metrics
        with open(
            os.path.join(output_dir, "metrics.json"),
            "w",
        ) as f:
            json.dump(metrics, f, indent=2)

        # Configuration
        config = {
            "word2vec_vector_size": args.vector_size,
            "word2vec_window": args.window,
            "word2vec_min_count": args.min_count,
            "word2vec_epochs": args.epochs,
        }

        with open(
            os.path.join(output_dir, "training_config.json"),
            "w",
        ) as f:
            json.dump(config, f, indent=2)

        # Vertex AI supplies AIP_MODEL_DIR.
        aip_model_dir = os.environ["AIP_MODEL_DIR"]

        logger.info(
            "Uploading artifacts to %s",
            aip_model_dir,
        )

        model_path = aip_model_dir.replace(
            "gs://", ""
        )

        bucket_name, prefix = model_path.split("/", 1)

        client = storage.Client()
        bucket = client.bucket(bucket_name)

        for filename in os.listdir(output_dir):
            blob = bucket.blob(
                prefix.rstrip("/") + "/" + filename
            )
            blob.upload_from_filename(
                os.path.join(output_dir, filename)
            )

        if hypertune:
            hypertune.HyperTune().report_hyperparameter_tuning_metric(
                hyperparameter_metric_tag="f1",
                metric_value=float(f1),
            )

        logger.info("Training completed.")
        logger.info("Metrics: %s", metrics)


if __name__ == "__main__":
    main()


## 7. Package and upload the training code

In [ ]:
!tar -czf custom.tar.gz custom

PACKAGE_URI = f"{BUCKET}/word2vec-test/custom.tar.gz"

!gsutil cp custom.tar.gz "$PACKAGE_URI"

print("Training package uploaded to:", PACKAGE_URI)


## 8. Create the Vertex AI custom training job

This is the equivalent of the `CustomPythonPackageTrainingJob` section in the Confluence document.


In [ ]:
job = aiplatform.CustomPythonPackageTrainingJob(
    display_name="word2vec-xgboost-test",

    python_package_gcs_uri=PACKAGE_URI,

    python_module_name="trainer.task",

    container_uri=TRAIN_IMAGE,

    project=PROJECT_ID,

    location=REGION,

    staging_bucket=BUCKET,

    training_encryption_spec_key_name=CMEK,

    model_encryption_spec_key_name=CMEK,
)

print("Training job object created.")


## 9. Submit the training job

For this smoke test we use one replica and a small machine.


In [ ]:
MODEL_DIR = f"{BUCKET}/word2vec-test/model"

model = job.run(
    args=[
        "--model-dir",
        MODEL_DIR,

        "--dataset-data-url",
        DATASET_URI,

        "--vector-size",
        "50",

        "--window",
        "3",

        "--min-count",
        "1",

        "--epochs",
        "10",

        "--xgb-rounds",
        "30",
    ],

    replica_count=1,

    machine_type="n1-standard-4",

    service_account=USE_CASE_SERVICE_ACCOUNT,

    sync=False,
)

print("Training job submitted.")


## 10. Check training status

Run this cell repeatedly until the job succeeds.


In [ ]:
print("Job state:", job.state)


## 11. Wait for completion

Use this only when you are ready to wait for the Vertex AI job to finish.


In [ ]:
job.wait()
print("Final job state:", job.state)


## 12. Check the model artifacts in GCS

The expected artifacts are:

- `model.bst` — XGBoost model
- `word2vec.model` — Word2Vec model
- `metrics.json` — evaluation metrics
- `training_config.json` — training configuration


In [ ]:
!gsutil ls "$MODEL_DIR/"


## 13. Read the evaluation metrics

In [ ]:
!gsutil cat "$MODEL_DIR/metrics.json"


## 14. Check the model registered by Vertex AI

In [ ]:
print("Model resource:")
print(model.resource_name)


## 15. Smoke-test the Word2Vec model locally

This confirms that the Word2Vec artifact can be loaded and used.


In [ ]:
!gsutil cp "$MODEL_DIR/word2vec.model" ./word2vec.model


In [ ]:
from gensim.models import Word2Vec

w2v = Word2Vec.load("word2vec.model")

print("Word2Vec vocabulary size:", len(w2v.wv))
print("Vector size:", w2v.vector_size)

word = "customer"

if word in w2v.wv:
    print("\nExample vector for 'customer':")
    print(w2v.wv[word])

    print("\nSimilar words:")
    print(w2v.wv.most_similar(word, topn=5))


# Expected result

If the pipeline is working, you should have successfully demonstrated:

1. Sample data created in the notebook.
2. Data uploaded to GCS.
3. Custom training package created.
4. Package uploaded to GCS.
5. Vertex AI custom training job submitted.
6. Word2Vec trained remotely.
7. XGBoost trained remotely using the document vectors.
8. `model.bst` and `word2vec.model` written to GCS.
9. Metrics written to `metrics.json`.
10. Model registered by Vertex AI.

**Do not use the 50-row accuracy/F1 as a model-quality result. This is only a technical pipeline test.**
